# Predicting Aircraft Remaining Useful Life (RUL)

In this laboratory, you will implement a full machine learning pipeline, from preprocessing to model evaluation. 

The dataset we use is a well-known one: the *Commercial Modular Aero-Propulsion System Simulation* or [CMAPSS](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data). It contains comprehensive information on aircraft engine performance throughout their lifecycles. It is a widely used benchmark dataset of **synthetic** data in the field of predictive maintenance.

The idea is to use ML to predict the **Remaining Useful Life** (RUL) of aircraft components based on their historical performance data. Even if synthetic, the dataset may present real-world challenges (e.g., missing values, sensor noise, useless features). The data exploration part will be crucial to understand the dataset and extract the most relevant features.

Some explanations of the dataset is available at the link above.

## Goals
The goal of this lab is to guide students towards a higher level of autonomy when dealing with ML problems, particularly regression-based predictive maintenance problems.

This document provides just the skeleton of your program, reminding you of the main steps to be accomplished. Feel free to use any information available as well as code from old labs.

At the end of this lab, you will be able to:
- Develop a full Machine Learning pipeline starting from a skeleton.
- Train, tune, and properly evaluate different ML models (k-nn, decision tree, and random forest).

## Methodology

### Data Acquisition
Utilize the NASA C-MAPSS dataset, specifically subset **FD001**, which contains:
- **Train trajectories**: 100
- **Test trajectories**: 100
- **Operating Conditions**: ONE (Sea Level)
- **Fault Modes**: ONE (HPC Degradation)

### Feature Engineering
Extract relevant features from the raw data, such as sensor readings, operating conditions, and maintenance history.

### Model Selection and Training
Experiment with various regression and classification models, including but not limited to:
- k-Nearest Neighbors (k-NN)
- Decision Tree
- Random Forest
- [Optional] Naive Bayes
- [Optional] Support Vector Machine (SVM)
- [Optional] ...

### Model Evaluation
Assess the performance of each model using appropriate metrics, such as:
- **Mean Absolute Error (MAE)**
- **Root Mean Squared Error (RMSE)**
- **R-squared Score**

By the end of this lab, you should have a trained ML model capable of predicting aircraft engine RUL and be able to compare different approaches to determine the most effective predictive maintenance strategy.

---

## 1 Data exploration 

### 1.1 Load the data
NOTE: don't worry if you see a warning after running the cell below (something like "*TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html from .autonotebook import tqdm as notebook_tqdm*"). It's normal.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

plt.style.use('bmh') # Use bmh style for plots

The .txt files used for traing, validation and test come without headers. Here below we provide the headers for the columns. 

In [ ]:
column_names = ['engine', 'time', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sm_{i}' for i in range(1, 22)]

The first five columns are:
1) Unit number (Engine number)
2) Time, in cycles
3) Operational setting 1
4) Operational setting 2
5) Operational setting 3

The operational settings (columns 3 to 5) are parameters that influence the engine's performance and operating conditions. While the dataset documentation does not specify the exact nature of these settings, they typically represent variables such as altitude, throttle resolver angle, and Mach number, which are known to substantially affect engine performance. These settings are included in the data to provide context for the sensor measurements and to facilitate the development of prognostic models.  As we do not know the exact nature of these settings, we will treat them as continuous variables and we will need to check their importance in the feature selection process.

The remaining columns are sensor measurements (therefore `sm`). Below, we provide a dictionary that maps the *sensor measurements* columns to their names that we got from the paper accompanying the dataset. If you are interested in the details of the sensors, you can check the paper (these data comes from Table 2). 

```python

In [ ]:
# assign names to columns, save in dict_list 
Sensor_dictionary={}
dict_list=[ "(Fan inlet temperature) (◦R)", #°R stands for Rankine scale. In general, scientists favor the Kelvin scale over Rankine. But hey, americans ;-)
"(LPC outlet temperature) (◦R)",
"(HPC outlet temperature) (◦R)",
"(LPT outlet temperature) (◦R)",
"(Fan inlet Pressure) (psia)",              # PSIA - PSI Absolute - Absolute pressure is measured relative to a full vacuum
"(bypass-duct pressure) (psia)",
"(HPC outlet pressure) (psia)",
"(Physical fan speed) (rpm)",
"(Physical core speed) (rpm)",
"(Engine pressure ratio(P50/P2)",
"(HPC outlet Static pressure) (psia)",
"(Ratio of fuel flow to Ps30) (pps/psia)",
"(Corrected fan speed) (rpm)",
"(Corrected core speed) (rpm)",
"(Bypass Ratio) ",
"(Burner fuel-air ratio)",
"(Bleed Enthalpy)",
"(Required fan speed)",
"(Required fan conversion speed)",
"(High-pressure turbines Cool air flow)",
"(Low-pressure turbines Cool air flow)" ]

i=1
for x in dict_list :
    Sensor_dictionary[f'sm_{i}']=x
    i+=1
Sensor_dictionary

In [ ]:
# We configure the dataset to be downloaded in the .data folder.
os.environ['KAGGLEHUB_CACHE'] = './data/'

In [ ]:
# (if not already in the data folder) we download the data from kaggle website
path = kagglehub.dataset_download("behrad3d/nasa-cmaps")

Give a look at the dataset in the `path` directory. The dataset is composed of multiple .txt files. You can use the `read_csv` function from the `pandas` library to load the data.

Some points to note:
- Data is separated by a single space, and the columns do not have headers. 
- Files have three prefixes: `train_`, `test_`, and `RUL_`.
    - The `RUL_` files contain the Remaining Useful Life values for the test trajectories. In other words, this is the ground truth for the test set, the value we want to predict.
    - The `train_` and `test_` files contain the sensor readings and operational settings for the engines. They do not have labels (a.k.a, target values or `y`). We will need to compute them.
- We will start our analyis using the `_FD001.txt` files. Later, we can extend our analysis to the other datasets, perhaps checking whether a model trained on one dataset can be used to predict the RUL of another dataset.

NOTE: you will get a warning message when you run the code below, but you can ignore it.

(*something like: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False [...]*)

In [ ]:
# Load the data into pandas dataframes

# os independent path building
data_train_path = os.path.join(path, "CMaps", "train_FD001.txt") # only training data, no label (RUL)
data_test_path = os.path.join(path, "CMaps", "test_FD001.txt") # only test data, no label (RUL)
y_test_path = os.path.join(path, "CMaps", "RUL_FD001.txt") # only RUL (label) for the test set

# load training data
train_df = pd.read_csv(data_train_path, sep=' ', header=None, names=column_names , index_col=False)

# load test data
test_data_df = pd.read_csv(data_test_path, sep=' ', header=None, names=column_names, index_col=False)
test_y_df = pd.read_csv(y_test_path , header=None , names=['RUL'])

In [ ]:
# we check the shape of the test data
print(test_data_df.shape)
print(test_y_df.shape)


In [ ]:
# We need to associate the y values to the test data.
# We will make prediction only for the last cycle of each engine, so we need to find the last cycle of each engine.
# Then we will associate the RUL value (test_y_df) to the last cycle of each engine.

# get the last cycle of each engine
last_cycle = test_data_df.groupby('engine').time.idxmax()

# associate the RUL value to the last cycle of each engine
test_data_df = test_data_df.loc[last_cycle]

# we check the shape of the test data
print(test_data_df.shape)
print(test_y_df.shape)


### 1.2 Explore the data

#### 1.2.1 Explore the training set

In [ ]:
# Display the first 10 rows of the training set
#TODO

In [ ]:
# Display the shape of the training set
#TODO

In [ ]:
# Display the information of the training set (e.g., datatype, number of non-null values, etc.)
#TODO

In [ ]:
# Display the statistics of the training set
#TODO

In [ ]:
# Check the number of unique values in each column of the training set
#TODO

**Question/Hint:**
- Some features have **one** unique value. What does that means in your opinion? We like them or not? Why?

**Answer**:
<details>
<summary><i>Click to reveal/hide the answer</i></summary>

- Features with only one unique value are constant features. They do not provide any information to the model and can be removed. In fact, they can even harm the model's performance. Therefore, we should (and we will) remove them from the dataset.

</details>

In [ ]:
# We check if there are any missing values in the training set
#TODO

#### 1.2.2 Label the data

As we mentioned above, the training set does not have a column for the Remaining Useful Life (RUL). This is because the RUL is a calculated value based on the number of cycles before the engine fails. The RUL is not given directly in the dataset, but it can be calculated using the following formula:

$$RUL = TTF - Cycles$$

Where:
- $RUL$ is the Remaining Useful Life
- $TTF$ is the Total Time to Failure (the number of cycles the engine runs before failure)
- $Cycles$ is the number of cycles the engine has already run


In [ ]:
# Backup the training set
# We are going to modify the training set. A good idea is to work on a copy of it. So, if we mess up, (and we probably will) we can always go back to the original data.

tr_df = train_df.copy() # tr_df is the copy of the training set

In [ ]:
# We define the function to calculate the Remaining Useful Life (RUL) of the engines
def calculate_rul(df):
    # We get the maximum cycle of each engine
    max_cycle = df.groupby('engine')['time'].transform("max")
    # We calculate the RUL
    df['RUL'] = max_cycle - df['time']
    return df

calculate_rul(tr_df)

Suggestion: the function above added a column to the dataframe. Give it a look and check if the values are what you expected.

In [ ]:
# max, or failure time for each engine, or max time cycle, that engine has worked.
failure_time = tr_df.groupby('engine')['RUL'].max()

In [ ]:
# plot the failure time for each engine and its distribution
#TODO

**Question**:
- Is the training set balanced or unbalanced? Why?

**Answer**:
<details>
<summary><i>Click to reveal/hide the answer</i></summary>
Most of the time, the analysis balanced/unbalanced is reserved for classification problems. Most of the existing methods for dealing with imbalanced data are only for classification problems, as well. However, we can observe that the dataset is "unbalanced" because the RUL values are not uniformly distributed (only a few engines have a high RUL). This can lead to a model that is biased towards predicting low RUL values. For further information you may want to check this <a href="https://towardsdatascience.com/strategies-and-tactics-for-regression-on-imbalanced-data-61eeb0921fca/">article</a>.

#### 1.2.3 Correlation analysis
To complete our data exploration, we will analyze the correlation between features and between the features and the target variable. This will help us understand which features are most relevant for predicting the RUL.

In [ ]:
# Print and or plot a correlation matrix
#TODO


**Questions**:
- Why is it important to know the type (numerical vs. categorical) of a feature?
- Why is it important to count the number of unique values in a feature?
- Why is it important to understand correlations? Is it good to have features highly correlated with each other? Is it good to have features highly correlated with the target? 
- Is our dataset unbalanced? Why is it important to know that? List possible solutions for dealing with an unbalanced dataset.
- Are there missing values in our dataset?
- Is this a regression or classification problem? Why?


**Answers**:
- To Do

## 2 Preprocessing the data

### 2.1 Drop useless columns

We should consider dropping columns that do not provide useful information for predicting the RUL:
- Columns with only one unique value
- Columns with very high correlation (>|0.95|) with other columns. We can just keep one of them.
- Columns with very low correlation with the target variable (<|0.1|)
- Other columns that do not provide useful information for predicting the RUL

NOTE: we will need to drop the same columns from the test set as well. We will do that in Section 5.

In [ ]:
# We drop the useless columns, if any
# We drop columns that have only one unique value
col_to_drop_unique_value = #TODO
col_to_drop_multicollinearity = #TODO
col_to_dop_low_correlation_with_target = #TODO
other = #TODO

# we put put all columns to drop in one list
col_to_drop = col_to_drop_unique_value + col_to_drop_multicollinearity + col_to_dop_low_correlation_with_target + other

# drop the columns
tr_df.drop(columns=col_to_drop, inplace=True)

**Question/Hint**:
- Should we drop the `time` column? Why?

**Answer**:
<details>
<summary><i>Click to reveal/hide the answer</i></summary>

The answer is **not** straightforward. The `time` column can be useful for predicting the RUL because the RUL is calculated based on the number of cycles before the engine fails. Of course, knowing that an engine has many cycles on his past is a good indicator of its remaining lifespan. However, the time column is also linearly correlated with the RUL, so it can introduce target leakage (i.e., the model can "cheat", learning the target from the features). In this case, we should drop the time column. The solution? Test with and without the time column and check the model's performance on a test set. If the model performs better without the time column, we should drop it.

</details>

### 2.2 Split the data into training and test set (if needed)

In [2]:
# Split the data into training and testing sets, if needed
#TODO

### 2.3 Imputation - fill missing values (if any)

In [4]:
# Impute missing values, if needed
#TODO

### 2.4 Separate the features X, from the label y (if needed)

In [5]:
# We separate the features and the target
#TODO

### 2.5 Feature scaling (if needed)
NOTE: Decision trees and random forests do not require feature scaling. However, k-NN does. Therefore, we will scale the features for all models.

In [6]:
# We rescale the features We use scikit-learn's scalers. Consider using the MinMaxScaler, the StandardScaler, or the RobustScaler.
#TODO


### 2.6 Encode the categorical features (if needed)

In [7]:
# We encode categorical features, if any
#TODO


### 2.7 Encode the Labels (if needed)

In [ ]:
# Encode the labels (label encoder)
#TODO

**Questions**:
- In this case, do we need to split the data into training and test sets? Why?
- Do we need to scale the features? Why?
- Do we need to encode the categorical features? Why?
- When do we need to encode the labels? Can you provide an example of a model that requires encoded labels?

All these "why" questions are very important to understand the reasons behind each step in the preprocessing phase. This understanding will help you to know when to apply each step in future projects.

**Answers**:
- To Do


## 3 Model Selection and Training

Now it's time to train and evaluate some models!

The code below is structured to allow easy experimentation with different models and comparison of their performance.

We will leverage [GridSearchCV](https://scikit-learn.org/stable/modules/grid_search.html) to optimize the hyperparameters of the models. GridSearchCV systematically tests various combinations of hyperparameters and uses cross-validation to assess the performance of each combination, ensuring a robust selection of the best configuration.

In [ ]:
# Import libraries (we will use GridSearchCV to find the best hyperparameters for each model)
from sklearn.model_selection import GridSearchCV

In [ ]:
# Define a function to find the best hyperparameters for a model
# Remember, the scoring depends on the model (e.g., for regression, we use 'r2' or 'neg_mean_squared_error'; for classification, we may use 'accuracy' or 'f1', etc.)

# Return the model with the best hyperparameters for the selected scoring (the scoring defined here will be used to optimize the hyperparameters)
def find_best_hyperparameters(regressor, param_grid, X_train, y_train, scoring = 'r2'):
    # create the grid search object
    grid_search = GridSearchCV(regressor, param_grid, cv=5, scoring=scoring, return_train_score=True, n_jobs=-1, verbose=2) # use all the cores of the CPU
    # find the best hyperparameters
    grid_search.fit(X_train, y_train)
    # print the best hyperparameters
    print("Best hyperparameters: {}".format(grid_search.best_params_))
    # print the best score
    print("Best score (r2): {:.2f}".format(grid_search.best_score_))
    # return the model with the best hyperparameters
    return grid_search.best_estimator_

In [ ]:
# train and evaluate the model with the best hyperparameters using cross-validation

from sklearn.model_selection import cross_validate
def print_cross_validation_results(regressor, X_train, y_train):
    # cross validate the model
    scoring = ['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error']
    cv_results = cross_validate(regressor, X_train, y_train, cv=5, scoring = scoring, n_jobs=-1, return_train_score=True)
    # Print the results
    print("Train R^2: {:.2f}".format(cv_results['train_r2'].mean()))
    print("Validation R^2: {:.2f}".format(cv_results['test_r2'].mean()))
    print("Train MAE: {:.2f}".format(-cv_results['train_neg_mean_absolute_error'].mean()))
    print("Validation MAE: {:.2f}".format(-cv_results['test_neg_mean_absolute_error'].mean()))
    print("Train RMSE: {:.2f}".format(-cv_results['train_neg_root_mean_squared_error'].mean()))
    print("Validation RMSE: {:.2f}".format(-cv_results['test_neg_root_mean_squared_error'].mean()))

*NOTE: For ALL models below, we will use these two functions to perform the grid search and print the results.*

### 3.1 Train a k-Nearest Neighbors (k-NN) model
(Feel free to use SVM instead of k-NN if you want to)

In [ ]:
# Import the k-nn classifier
from sklearn.neighbors import KNeighborsRegressor

# Define the parameter grid to explore
param_grid = {'n_neighbors': [1, 10, 20]}

# Create the regressor
knn_reg = #TODO

# Find the best hyperparameters
knn_reg = #TODO

# Print the cross validation results
#TODO

### 3.2 Train a Decision Tree model

In [ ]:
# Import the decision tree classifier
from sklearn.tree import DecisionTreeRegressor

# Define the parameter grid to explore
param_grid = #TODO
# Create the regressor
dt_reg = #TODO

# Find the best hyperparameters
dt_reg = #TODO

# Print the cross validation results
#TODO

**Questions**:
Describe the meaning of the hyperparameters:
- `max_depth`. What's the meaning of this hyperparameter? What happens if we set it to `None`?
- `min_samples_split`
- `min_samples_split`

**Answers**:
- To Do

### 3.3 Train a Random Forest model

In [ ]:
# Import the random forest classifier
from sklearn.ensemble import RandomForestRegressor

# Define the parameter grid to explore
param_grid = {'n_estimators': [10, 100], 'max_depth': [None, 1, 3], 'min_samples_split': [2, 4], 'min_samples_leaf': [1]}

# Create the regressor
rf_reg = #TODO

# Find the best hyperparameters
rf_reg = #TODO

# Print the cross validation results
#TODO

# Rank and print the feature importances with its corresponding feature name
#TODO


**Questions**:
- Describe the meaning of the hyperparameters `n_estimators`. What are the possible values for this hyperparameter? An high value of `n_estimators` is always better?

**Answers**:
- To Do

### 3.4 learning_curve

Plotting a learning curve could help to see if the models are overfitting and/or if getting more data may help.

In [ ]:
# Plot the learning curves for k-NN, Decision Tree, and Random Forest in separate subplots
# Keep the same y-axis for all the subplots to better compare the models
# Use the r2 score as the scoring metric. The higher the r2 score, the better the model.
#TODO



In [ ]:
# We do the same with MAE. The lower the MAE, the better the model.
#TODO


**Questions**:
- What is a learning curve? What information can you extract from it?
- Which deductions can you make from the learning curve?
- What do you observe in your learning curves?

**Answers**:
- To Do

## 4 Model Evaluation

Finally, we evaluate the models using appropriate metrics on the test set. As above, we will use the following metrics to evaluate the models:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared Score

### 4.1 Evaluation of the three models on the test set

In [ ]:
# Evaluate the best models on the test set
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# We rescale the test set
#TODO

# We predict the RUL
#TODO

# We evaluate the models
#TODO


**Questions**:
- What is the best model? Why?
- Why we reused the scaler from the training set to transform the test set? Why is this important?
- Are you satisfied with the model's performance? Why?
- What could be say from metrics such as MAE and RMSE?
- What could be say from the R-squared score?

**Answers**:
- To Do

### 4.2 (Bonus) Prediction on a single sample

We use the best model to predict the RUL of a single sample. This is just to show how to use the model to make predictions on new data.

In [8]:
# We make prediction on a random sample of the test set and compare the predictions with the true values.
# We use the Random Forest regressor because it has the best performance.

#TODO


## 5 Hyperparameter tuning with Genetic Algorithms

Modify the code above to use Genetic Algorithms for hyperparameter tuning instead of Grid Search.

We suggest you write a function structured similarly to the grid search function above. Feel free to use a GA library such as [DEAP](https://deap.readthedocs.io/en/master/).

In the cell below, we provide some hints to get you started.

In [ ]:
def evalModel(individual, clf, X, y, scoring='f1', cv=5):
    # Individual is a list of (param_name, value)

    # Set hyperparameters into the classifier


    # Perform cross-validation
  

    # Return mean score as a tuple (DEAP requires tuples for fitness)
    return (scores.mean(),)


# -----------------------------
# Individual Generator
# -----------------------------
def create_individual(param_grid):
    # Todo: Randomly sample one value per hyperparameter from the grid.

    return individual


# -----------------------------
# Main GA Function
# -----------------------------
def GA_search(clf, param_grid, X_train, y_train, scoring='f1', ngen=10, pop_size=50):


    # -----------------------------
    # DEAP Setup
    # -----------------------------

    # Fitness function (we want to maximize score)
    

    # Individuals are lists of (param_name, value)
    

    # -----------------------------
    # Run Evolution
    # -----------------------------



# -----------------------------
# GA Execution Loop
# -----------------------------
def run_evolution(toolbox, ngen=10, pop_size=50):


    # Evaluate initial population
   


        # Select, clone, crossover, and mutate
 

        # Apply crossover


        # Apply mutation


        # Re-evaluate only invalid individuals


        # Replace population
        
    # Etc...


## Conclusion
In this lab, we implemented a full machine learning pipeline to predict the Remaining Useful Life (RUL) of aircraft components based on their historical performance data. We explored the dataset, preprocessed the data, trained different models, and evaluated their performance. We used the Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared Score to evaluate the models. We also discussed the importance of understanding the data, preprocessing the data, and evaluating the models. We hope this lab helped you gain a better understanding of machine learning and predictive maintenance.

Now, you have a full pipeline to do some simple predictive maintenance and deal with regression problems.